# PVS Thailand analysis

This notebook is the entry point for exploring the approved September results and, with authorized private inputs, reproducing the analysis. All analysis logic lives in the `pvs` package.

Data analyst: **Peerasit Sitthirat (พีรสิชฌ์ สิทธิรัตน์)**, Faculty of Medicine Ramathibodi Hospital, Mahidol University. See the repository README for project organizations and funding.

## Setup

From the repository root, install the environment with `python -m pip install -r requirements-lock.txt` and `python -m pip install -e '.[notebook]'`. Select that environment as your notebook kernel.

By default this notebook only reads approved, aggregate release files. Set `PVS_PROJECT_ROOT` before importing `pvs` if your workspace is elsewhere.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display, Image
from pvs.paths import ROOT, OUTPUT

REPRODUCE = False
RELEASE = ROOT / 'results' / '2026-09'

## Optional full reproduction

Set `REPRODUCE = True` above only when the private inputs described in `data/README.md` are available. This runs the same `python -m pvs` workflow as the command line and overwrites working files in `output/`. The existing cleaner can update working copies of the dictionary and recoding maps.

Detailed execution logs stay in the ignored `output/notebook_run.log`; participant-level frames are never displayed here.

In [ ]:
if REPRODUCE:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    log_path = OUTPUT / 'notebook_run.log'
    with log_path.open('w', encoding='utf-8') as log:
        result = subprocess.run(
            [sys.executable, '-m', 'pvs', '--project-root', str(ROOT)],
            stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT,
        )
    if result.returncode:
        raise RuntimeError(f'Analysis failed; inspect the local log: {log_path}')
    print('Full reproduction completed. Working results are in output/.')
else:
    print('Viewing the approved September release; private inputs are not required.')

VIEW = OUTPUT if REPRODUCE else RELEASE

## Analytic cohort

The cohort flow records the denominators used in the analysis. Values are read from exported tables; no estimates are recalculated in this notebook.

In [ ]:
display(pd.read_csv(VIEW / 'tables' / 'T01_cohort_flow.csv'))

## Provider profiles

Compare the four usual-source-of-care groups using the exported profile table and figure.

In [ ]:
display(pd.read_csv(VIEW / 'tables' / 'T20_profile_by_group.csv'))
display(Image(filename=str(VIEW / 'figures' / 'Figure 1 - Profile by provider group.png')))

## Adjusted results

These are associations, not causal effects. Consult the model-note tables and report for denominator rules, sampling limitations, confidence intervals, and multiplicity adjustment.

In [ ]:
display(pd.read_csv(VIEW / 'tables' / 'T59_core_outcomes_adjusted.csv'))

## Report and reproducibility

The approved report is in `results/2026-09/report.docx`. A fresh report is produced under `output/` when full reproduction is enabled. See `docs/reproduction.md` and `docs/validation.md` for the workflow and verification record.

Before committing this notebook, clear all outputs and execution counts. `python tools/verify_release.py` checks this alongside the approved release files.